# 7.1 Checklist tables

This notebook does the following:
    - Table 1: Bronze/Silver/Gold checklist question responses at baseline and endline, and lab-level change (improved/unimproved)

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config

In [2]:
# Load data (blank cells are genuinely missing, not the string "NA")
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False,
    na_values=[""]
)

## (1) Prepare data

The `_bl`/`_el` checklist columns are identical across a lab's BL and EL rows, so we keep only one row per lab. Only treatment labs did the checklist.

In [3]:
# Keep only labgroups with both BL and EL data
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
matched_labgroupids = labgroup_counts[labgroup_counts == 2].index

# Keep only BL rows
df_bl = df[df["labgroupid"].isin(matched_labgroupids) & (df["survey"] == "BL")].copy()
df_treatment = df_bl[df_bl["treated"] == 1].copy()

In [4]:
# Restrict to labs that answered at least 1 checklist q at BL and EL

# Check how many labs answered at least 1 checklist q at BL
checklist_cols_bl = (
    [f"bronze_q_{i}_bl" for i in range(1, 17)]
    + [f"silver_q_{i}_bl" for i in range(1, 19)]
    + [f"gold_q_{i}_bl" for i in range(1, 16)]
)

df_treatment["answered_any_bl"] = df_treatment[checklist_cols_bl].notna().any(axis=1)
print(f"Number of labs that answered at least 1 checklist question at BL: {df_treatment['answered_any_bl'].sum()}")

# Check how many labs answered at least 1 checklist q at EL
checklist_cols_el = (
    [f"bronze_q_{i}_el" for i in range(1,17)]
    + [f"silver_q_{i}_el" for i in range(1,19)]
    + [f"gold_q_{i}_el" for i in range(1,16)]
)
df_treatment["answered_any_el"] = df_treatment[checklist_cols_el].notna().any(axis=1)
print(f"Number of labs that answered at least 1 checklist question at EL: {df_treatment['answered_any_el'].sum()}")

# Restrict to labs that answered at least 1 checklist q at BL and EL
df_checklist_sample = df_treatment[df_treatment["answered_any_bl"] & df_treatment["answered_any_el"]].copy()
print(f"Number of labs that answered at least 1 checklist question at both BL and EL: {len(df_checklist_sample)}")

Number of labs that answered at least 1 checklist question at BL: 51
Number of labs that answered at least 1 checklist question at EL: 51
Number of labs that answered at least 1 checklist question at both BL and EL: 48


In [5]:
# Item labels: keys are the column stem shared by the _bl/_el pair
item_labels = {}
item_labels.update({f"bronze_q_{i}": f"Bronze Q{i}" for i in range(1, 17)})
item_labels.update({f"silver_q_{i}": f"Silver Q{i}" for i in range(1, 19)})
item_labels.update({f"gold_q_{i}": f"Gold Q{i}" for i in range(1, 16)})

section_headers = {
    "Bronze Checklist": [f"bronze_q_{i}" for i in range(1, 17)],
    "Silver Checklist": [f"silver_q_{i}" for i in range(1, 19)],
    "Gold Checklist": [f"gold_q_{i}" for i in range(1, 16)],
}

# Question categories
item_categories = {}
# Bronze qs
for i in range(1, 7):
    item_categories[f"bronze_q_{i}"] = "General Lab"
for i in range(7, 10):
    item_categories[f"bronze_q_{i}"] = "Offices \& Travel"
item_categories["bronze_q_10"] = "Cold Storage"
for i in range(11, 16):
    item_categories[f"bronze_q_{i}"] = "Chemistry"
item_categories["bronze_q_16"] = "Fume Cupboards"
# Silver qs
for i in range(1, 9):
    item_categories[f"silver_q_{i}"] = "General Lab"
for i in range(9, 12):
    item_categories[f"silver_q_{i}"] = "Offices \& Travel"
item_categories["silver_q_12"] = "Cold Storage"
for i in range(13, 18):
    item_categories[f"silver_q_{i}"] = "Chemistry"
item_categories["silver_q_18"] = "Fume Cupboards"
# Gold qs
for i in range(1, 7):
    item_categories[f"gold_q_{i}"] = "General Lab"
for i in range(7, 10):
    item_categories[f"gold_q_{i}"] = "Offices \& Travel"
item_categories["gold_q_10"] = "Cold Storage"
for i in range(11, 15):
    item_categories[f"gold_q_{i}"] = "Chemistry"
item_categories["gold_q_15"] = "Fume Cupboards"

## (2) Checklist summary table

Columns: Question, Category, then Baseline (% Yes / I don't know / No / N/A), Endline (same four), Improved (\%) (the share of labs with a substantive Yes/No/I-don't-know answer at both waves whose BL answer was No or I don't know and whose EL answer was Yes), and N applicable (the number of labs with a substantive answer at both waves).

N/A groups together "Not applicable" and "didn't answer this question" -- both mean no substantive answer. The sample is fixed to labs that answered at least 1 checklist question at both BL and EL (48 labs).

In [6]:
improved_from = ("No", "I don't know")
improved_to = "Yes"
decimals = 0

col1_width = "3cm" # width of question col
col2_width = "3cm" # width of category col
coln_width = "1.5cm" # width of data cols
n_width = "1.3cm" # width of N applicable col
indent = r"\hspace{0.3cm} "


def fmt_pct(val, d):
    if val != val:  # NaN
        return ""
    return f"${val:.{d}f}$"


def fmt_n(val):
    return f"${int(val)}$" if val == val else ""

In [7]:
header_for_item = {}
for header, section_items in section_headers.items():
    for it in section_items:
        header_for_item[it] = header
emitted_headers = set()

n_data_cols = 10  # Baseline (4) + Endline (4) + Improved (1) + N applicable (1)
widths = [coln_width] * 4 + [coln_width] * 4 + [coln_width] + [n_width]
col_spec = f"@{{}}L{{{col1_width}}}L{{{col2_width}}}" + "".join(f"C{{{w}}}" for w in widths)

lines = []
lines.append(f"\\begin{{tabular}}{{{col_spec}}}")
lines.append(r"\hline")
lines.append(r"\addlinespace[0.2cm]")

# Top-level column groups (Question, Category have no group header)
lines.append(
    r" & & \multicolumn{4}{c}{Baseline} & \multicolumn{4}{c}{Endline} & & \\"
)
lines.append(
    r"\cmidrule(lr){3-6} \cmidrule(lr){7-10}"
)

# Sub-column labels
sub_labels_1 = ["Yes", "I don't", "No", "N/A"] * 2 + ["Improved", "N"]
lines.append("Question & Category & " + " & ".join(sub_labels_1) + r" \\")
sub_labels_2 = ["(\%)", "know (\%)", "(\%)", "(\%)"] * 2 + ["(\%)", "applicable"]
lines.append(" & & " + " & ".join(sub_labels_2) + r" \\")
lines.append(r"\hline")
lines.append(r"\addlinespace[0.2cm]")

# N/A groups "Not applicable" and didn't answer q
# Substantive responses are "Yes", "I don't know", and "No"
substantive = ("Yes", "I don't know", "No")
n_fixed = len(df_checklist_sample)

for item in item_labels:
    header = header_for_item.get(item)
    if header is not None and header not in emitted_headers:
        lines.append(f"\\multicolumn{{{n_data_cols + 2}}}{{@{{}}l}}{{{header}}} \\\\")
        lines.append(r"\addlinespace[0.1cm]")
        emitted_headers.add(header)

    label_str = item_labels[item]
    if header is not None:
        label_str = f"{indent}{label_str}"
    category_str = item_categories.get(item, "")

    # Get BL and EL responses for this question
    bl = df_checklist_sample[f"{item}_bl"]
    el = df_checklist_sample[f"{item}_el"]

    # Determine which responses are substantive at BL and EL for this question
    is_substantive_bl = bl.isin(substantive)
    is_substantive_el = el.isin(substantive)

    # Calculate percentages for each category at BL and EL
    bl_pcts = [(bl == cat).sum() / n_fixed * 100 for cat in substantive] + [
        (~is_substantive_bl).sum() / n_fixed * 100
    ]
    el_pcts = [(el == cat).sum() / n_fixed * 100 for cat in substantive] + [
        (~is_substantive_el).sum() / n_fixed * 100
    ]

    # Define improvement - changed from "No" or "I don't know" at BL to "Yes" at EL
    # Share improved is of labs with a substantive answer at both BL and EL
    both_substantive = is_substantive_bl & is_substantive_el
    n_change = int(both_substantive.sum())
    if n_change > 0:
        improved = both_substantive & bl.isin(improved_from) & (el == improved_to)
        pct_improved = improved.sum() / n_change * 100
    else:
        pct_improved = float("nan")

    row_vals = (
        [fmt_pct(v, decimals) for v in bl_pcts]
        + [fmt_pct(v, decimals) for v in el_pcts]
        + [fmt_pct(pct_improved, decimals), fmt_n(n_change)]
    )
    lines.append(f"{label_str} & {category_str} & " + " & ".join(row_vals) + r" \\")

lines.append(r"\addlinespace[0.2cm]")
lines.append(r"\hline")
lines.append(r"\end{tabular}")

table = "\n".join(lines)

In [8]:
out_dir = config.OUTPUT / "9_Checklist_Tables"
out_dir.mkdir(parents=True, exist_ok=True)
table_path = out_dir / "checklist_table.tex"
_ = table_path.write_text(table)